
I###dentity Resolution Example using Vector Embeddings from a Doc2Vec MD.
 Siskmodel a Doc2Vec model
aDa
a,id C.v Sisk, 2026/0303/03-03-03v 2
is notebook requires a pre-trained doc2vec model, which you can download from here (quite large):
https://mega.nz/file/S2wxxArK#akoKdYl4SaO2JX29AHDWQ8skbDGnxrih9-mViK_ezWA

In [ ]:
#! pip install pandas
#! pip install scikit-learn
#! pip install gensim

In [81]:
import pandas as pd

df_inputdata = pd.read_csv('sample-data-messy_200.csv')

df_inputdata.shape

(200, 6)

In [82]:
# Calculate vector embeddings using a pretrained Doc2Vec model instead of an LLM
# Tokenize the text before calling the model
from gensim.models.doc2vec import Doc2Vec

# Load the pretrained doc2vec model
doc2vec_model = Doc2Vec.load("doc2vec_wikipedia_dm.model")

# Function to generate embeddings for a row
from gensim.utils import simple_preprocess
def generate_doc2vec_embedding(row):
    # Exclude 'row_id' and 'true_id' columns and concatenate other columns
    text = " ".join(str(value) for key, value in row.items() if key not in ['row_id', 'true_id'])
    # Tokenize using gensim's simple_preprocess
    tokens = simple_preprocess(text)
    # Infer vector using the doc2vec model from tokenized text
    vector = doc2vec_model.infer_vector(tokens)
    return vector

# Apply the embedding function to each row
df_inputdata['embedding'] = df_inputdata.apply(generate_doc2vec_embedding, axis=1)

print("df_inputdata.shape:", df_inputdata.shape)

embedding_dimension = len(df_inputdata['embedding'].iloc[0])
print(f"The embedding dimension count is: {embedding_dimension}")

df_inputdata.shape: (200, 7)
The embedding dimension count is: 200


In [ ]:
# Calculate vector embeddings using a pretrained Doc2Vec model instead of an LLM
# Not tokenized
from gensim.models.doc2vec import Doc2Vec

# Load the pretrained doc2vec model
doc2vec_model = Doc2Vec.load("doc2vec_wikipedia_dm.model")

# Function to generate embeddings for a row
def generate_doc2vec_embedding(row):
    # Exclude 'row_id' and 'true_id' columns and concatenate other columns
    text = " ".join(str(value) for key, value in row.items() if key not in ['row_id', 'true_id'])
    # Infer vector using the doc2vec model
    vector = doc2vec_model.infer_vector(text.split())
    return vector

# Apply the embedding function to each row
df_inputdata['embedding'] = df_inputdata.apply(generate_doc2vec_embedding, axis=1)

print("df_inputdata.shape:", df_inputdata.shape)

embedding_dimension = len(df_inputdata['embedding'].iloc[0])
print(f"The embedding dimension count is: {embedding_dimension}")

df_inputdata.shape: (200, 7)
The embedding dimension count is: 200


In [83]:
# L2-normalize vector embeddings and store in a new column
target_df = df if ("df" in globals() and "embedding" in df.columns) else df_inputdata

target_df["l2n_embedding"] = target_df["embedding"].apply(
    lambda v: v / np.linalg.norm(v) if np.linalg.norm(v) != 0 else v
)

In [84]:
# Using the inputdata above, construct a pairwise dataframe for every unique combination
from itertools import combinations

# Create lists of row_id and true_id
row_ids = df_inputdata['row_id'].tolist()
true_ids = df_inputdata['true_id'].tolist()

# Helper function to get field value, replacing NaN with empty string
def get_field(row, col):
    val = df_inputdata.iloc[row][col]
    return '' if pd.isna(val) else str(val)

# Create all pairwise combinations
pairwise_data = []
for i in range(len(df_inputdata)):
    for j in range(len(df_inputdata)):
        if i != j:  # Don't pair a row with itself
            # Concatenate name, email, address, and phone for both rows
            data1 = ' | '.join([
                get_field(i, 'name'),
                get_field(i, 'email'),
                get_field(i, 'address'),
                get_field(i, 'phone')
            ])
            data2 = ' | '.join([
                get_field(j, 'name'),
                get_field(j, 'email'),
                get_field(j, 'address'),
                get_field(j, 'phone')
            ])
            pairwise_data.append({
                'row_id1': row_ids[i],
                'true_id1': true_ids[i],
                'data1': data1,
                'embedding1': df_inputdata['l2n_embedding'].iloc[i],
                'row_id2': row_ids[j],
                'true_id2': true_ids[j],
                'data2': data2,
                'embedding2': df_inputdata['l2n_embedding'].iloc[j]
            })

df_pairwise = pd.DataFrame(pairwise_data)

df_pairwise.shape

(39800, 8)

In [85]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2
9225,row-0077,id-0045,"Parker, Jonathan | jparker@mail.example.org | ...","[0.098160826, -0.041315995, 0.0042327293, 0.02...",row-0120,id-0087,"Newman, Sadie | snewman@mail.example.org | | ...","[0.066733964, -0.045802902, -0.07055401, 0.079..."
6477,row-0053,id-0009,"Anderson, Charlotte | canderson@mail.example.o...","[0.047853302, -0.08082581, 0.036344517, 0.0351...",row-0182,id-0037,"Luke Nelson | | 37 Summit St, Springfield, IL...","[0.08644128, -0.01878978, -0.032575317, 0.1009..."
607,row-0005,id-0011,"Jackson, Henry | hjackson@mail.example.org | 1...","[0.067753635, -0.05443762, -0.09221586, -0.003...",row-0019,id-0053,"Reed, Adrian | areed@mail.example.org | 5353 5...","[0.0714798, -0.06423189, 0.005062989, 0.055426..."


In [86]:
# For each pairwise row, calculate cosine similarity between embedding1 and
# embedding2, and store them in a new column called "cosine_similarity"
import numpy as np

def cosine_sim(v1, v2):
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    return float(np.dot(v1, v2) / denom) if denom != 0 else np.nan

df_pairwise["cosine_similarity"] = [
    cosine_sim(v1, v2)
    for v1, v2 in zip(df_pairwise["embedding1"], df_pairwise["embedding2"])
]


In [87]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity
19790,row-0169,id-0086,"Fox, Eleanor | efox@mail.example.org | 8686 86...","[0.06217692, -0.005437222, -0.0101495655, 0.04...",row-0154,id-0030,Lvei Lopez | levi.lopez1@example.net | 3031 30...,"[0.02242056, -0.015494327, -0.0053849476, 0.03...",0.481247
15800,row-0136,id-0084,Jsamine Burke | jasmine.burke6@example.net | 8...,"[0.03711487, -0.032866288, 0.040152173, 0.0890...",row-0138,id-0032,"Scott, Mateo | mscott@mail.example.org | 3232 ...","[0.07995038, -0.019467577, -0.069873914, 0.050...",0.412809
36279,row-0275,id-0097,"Shaw, Miranda | mshaw@mail.example.org | | (5...","[0.040008932, -0.08043552, -0.037885945, 0.043...",row-0101,id-0033,Oewn Green | owen.green4@example.net | 3332 33...,"[0.073241554, -0.06817478, -0.013553508, 0.080...",0.444835


In [88]:
# Min-max normalize cosine similarity to a 0-100 scale and store that as the score
min_sim = df_pairwise["cosine_similarity"].min()
max_sim = df_pairwise["cosine_similarity"].max()

if max_sim == min_sim:
    df_pairwise["score"] = 100.0
else:
    df_pairwise["score"] = (
        (df_pairwise["cosine_similarity"] - min_sim) / (max_sim - min_sim) * 100
    )

# round score to 2 decimal places
df_pairwise["score"] = df_pairwise["score"].round(2)

In [89]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
39603,row-0300,id-0020,Dnaiel Rodriguez | daniel.rodriguez5@example.n...,"[-0.048995197, -0.07094717, 0.0068141245, 0.05...",row-0004,id-0030,"Lopez, Levi | llopez@mail.example.org | 3030 3...","[0.10957985, -0.027743917, 0.0320014, 0.018920...",0.529572,65.31
39610,row-0300,id-0020,Dnaiel Rodriguez | daniel.rodriguez5@example.n...,"[-0.048995197, -0.07094717, 0.0068141245, 0.05...",row-0015,id-0069,"Parker Fisher | | 69 Cedarview Ave., Springfi...","[0.084184535, 0.010511725, -0.0042594015, 0.09...",0.551303,67.13
20773,row-0175,id-0043,Atnhony Phillips | anthony.phillips0@example.n...,"[0.010364734, 0.011373941, 0.023445006, 0.0579...",row-0131,id-0075,"Stella Pierce | | 75 Sunrise Blvd, Springfiel...","[0.025301788, -0.07590281, -0.051632743, 0.133...",0.579718,69.51


In [90]:
# Display rows where true_id1 equals true_id2
df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]


,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
142,row-0002,id-0080,"Chavez, Leah | lchavez@mail.example.org | 8080...","[0.074704766, -0.05440344, -0.01112992, 0.0485...",row-0224,id-0080,Laeh Chavez | leah.chavez2@example.net | 8081 ...,"[-0.01876111, -0.06115535, 0.013266617, 0.0291...",0.656711,75.97
313,row-0003,id-0013,"James Harris | | 13 Sycamore St, Springfield,...","[0.07161375, -0.039065775, -0.018786926, 0.130...",row-0190,id-0013,Jmaes Harris | james.harris5@example.net | 131...,"[0.03936246, -0.032717887, -0.008189653, 0.100...",0.659178,76.18
486,row-0004,id-0030,"Lopez, Levi | llopez@mail.example.org | 3030 3...","[0.10957985, -0.027743917, 0.0320014, 0.018920...",row-0154,id-0030,Lvei Lopez | levi.lopez1@example.net | 3031 30...,"[0.02242056, -0.015494327, -0.0053849476, 0.03...",0.680284,77.95
647,row-0005,id-0011,"Jackson, Henry | hjackson@mail.example.org | 1...","[0.067753635, -0.05443762, -0.09221586, -0.003...",row-0087,id-0011,"Henry Jackson | | 11 Willow Dr, Springfield, ...","[0.05524857, -0.022363631, -0.11831596, 0.0727...",0.666451,76.79
868,row-0008,id-0041,Rayn Roberts | ryan.roberts5@example.net | | ...,"[-0.00752697, -0.045742612, 0.003570286, 0.061...",row-0121,id-0041,"Roberts, Ryan | rroberts@mail.example.org | 41...","[0.11921263, 0.015129293, 0.072077215, 0.03481...",0.456025,59.14
...,...,...,...,...,...,...,...,...,...,...
38954,row-0294,id-0018,"Alexander Robinson | | 18 Cypress Ave., Sprin...","[0.072105035, -0.061875906, -0.03640276, 0.054...",row-0232,id-0018,"Robinson, Alexander | arobinson@mail.example.o...","[0.11782559, -0.06938848, 0.004058262, -0.0250...",0.587618,70.18
39159,row-0296,id-0039,Claeb Mitchell | caleb.mitchell3@example.net |...,"[0.05108874, -0.018786876, 0.035904076, 0.0609...",row-0239,id-0039,"Mitchell, Caleb | cmitchell@mail.example.org |...","[0.10572626, -0.00488493, 0.01305144, 0.057241...",0.670934,77.17
39261,row-0298,id-0047,"Edwards, Thomas | tedwards@mail.example.org | ...","[0.011568987, -0.052451834, -0.062019467, 0.09...",row-0096,id-0047,Tohmas Edwards | thomas.edwards4@example.net |...,"[-0.00558999, -0.053776156, 0.00413471, 0.0190...",0.479547,61.11
39531,row-0299,id-0005,Aav Miller | ava.miller4@example.net | 566 5 B...,"[-0.0022369034, -0.10798561, 0.004460176, 0.04...",row-0207,id-0005,"Miller, Ava | amiller@mail.example.org | 567 5...","[0.07462457, -0.08857617, -0.010577778, 0.0130...",0.655628,75.88


In [111]:
# Examine the score spread where they were true matches

# Filter rows where true_id1 equals true_id2
matches = df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]
true_match_count = len(matches)

# Calculate statistics
count = len(matches)
min_score = matches['score'].min()
avg_score = matches['score'].mean()
max_score = matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 200
Min Score: 40.61
Avg Score: 70.98
Max Score: 89.14


In [110]:
# Examine the score spread where they are NOT true matches

# Filter rows where true_id1 does NOT equal true_id2
non_matches = df_pairwise[df_pairwise['true_id1'] != df_pairwise['true_id2']]
non_match_count = len(non_matches)

# Calculate statistics
count = len(non_matches)
min_score = non_matches['score'].min()
avg_score = non_matches['score'].mean()
max_score = non_matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 39600
Min Score: 0.0
Avg Score: 62.84
Max Score: 100.0


In [115]:
# Choose the matching score cutoff threshold that gets the most acceptable mix 
# of false positives and false negatives
#cutoff = 47.76  # This is the middle of the average scores...reasonable starting point
cutoff = 65.00
df_pairwise["match"] = (df_pairwise["score"] >= cutoff).astype(int)

# Calculate Precision & Recall as our accuracy metrics
# True positives: predicted match (1) and actually same person (true_id1 == true_id2)
true_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# False positives: predicted match (1) but actually different people (true_id1 != true_id2)
false_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] != df_pairwise["true_id2"])
]

# False negatives: predicted no match (0) but actually same person (true_id1 == true_id2)
false_negatives = df_pairwise[
    (df_pairwise["match"] == 0) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# True negatives: predicted no match (0) and actually different people (true_id1 != true_id2)
true_negatives = df_pairwise[
    (df_pairwise["match"] == 0) & (df_pairwise["true_id1"] != df_pairwise["true_id2"])
]


# Calculate Precision & Recall, plus F1 score
precision = len(true_positives) / (len(true_positives) + len(false_positives))
recall = len(true_positives) / (len(true_positives) + len(false_negatives))
F1_score = 2 * (precision * recall) / (precision + recall)

print("Non-LLM example...")
print("Generated true matches in sample data:", true_match_count)
print("Predicted matches:", (df_pairwise["match"] == 1).sum())
print("Generated true non-matches in sample data:", non_match_count)
print("Predicted non-matches:", (df_pairwise["match"] == 0).sum())
print("")
print(f"TRUE POSITIVES: {len(true_positives)}")
print(f"FALSE POSITIVES: {len(false_positives)}")
print(f"FALSE NEGATIVES: {len(false_negatives)}")
print(f"TRUE NEGATIVES: {len(true_negatives)}")
print("Check: TRUE positives + FALSE negatives = 200? ->", len(true_positives) + len(false_negatives))
print("")
print(f"PRECISION: {precision:.4f}  (What % of predicted matches were correct true matches?)")
print(f"RECALL: {recall:.4f}  (What % of true matches were correctly predicted?)")
print(f"F1 SCORE: {F1_score:.4f}  (Harmonic mean of precision and recall)") 

Non-LLM example...
Generated true matches in sample data: 200
Predicted matches: 17984
Generated true non-matches in sample data: 39600
Predicted non-matches: 21816

TRUE POSITIVES: 154
FALSE POSITIVES: 17830
FALSE NEGATIVES: 46
TRUE NEGATIVES: 21770
Check: TRUE positives + FALSE negatives = 200? -> 200

PRECISION: 0.0086  (What % of predicted matches were correct true matches?)
RECALL: 0.7700  (What % of true matches were correctly predicted?)
F1 SCORE: 0.0169  (Harmonic mean of precision and recall)
